In [7]:
import numpy as np

In [8]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = data.data          # Shape: (569, 30)
y = data.target        # Shape: (569,)

y = y.reshape(-1, 1)   # Shape: (569, 1)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
class Affine:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def forward(self, x):
        return x @ self.weight.T + self.bias



In [11]:
class Sigmoid:
    def __init__(self):
        pass

    def forward(self, x):
        return np.exp(x) / (np.exp(x) + 1)


In [12]:
class Softmax:
  def __init__(self):
    pass

  def forward(self, x):
    sumv = 0
    for i in range(len(x[0])):
        if x[0][i]< -100:
            x[0][i] = -100
        x[0][i] = np.exp(x[0][i])
        sumv += x[0][i]
    for i in range(len(x)):
        x[0][i] /= sumv
    return x



In [13]:
class MLP_b:
    def __init__(self):
        self.g = [[np.random.randn(8, 30) * 0.1, np.zeros((1, 8))],
                  [np.random.randn(8, 8) * 0.1, np.zeros((1, 8))],
                  [np.random.randn(8, 8) * 0.1, np.zeros((1, 8))],
                  [np.random.randn(8, 8) * 0.1, np.zeros((1, 8))],
                  [np.random.randn(1, 8) * 0.1, np.zeros((1, 1))]]
        self.update()
    def forward(self, x):
        for layer in self.layers:
            for module in layer:
                x =  module.forward(x)
        return x.astype(float)
    def update(self):
        self.affine1 = Affine(self.g[0][0], self.g[0][1])
        self.activation1 = Sigmoid()
        self.layer1 = [self.affine1, self.activation1]

        self.affine2 = Affine(self.g[1][0], self.g[1][1])
        self.activation2 = Sigmoid()
        self.layer2 = [self.affine2, self.activation2]

        self.affine3 = Affine(self.g[2][0], self.g[2][1])
        self.activation3 = Sigmoid()
        self.layer3 = [self.affine3, self.activation3]

        self.affine4 = Affine(self.g[3][0], self.g[3][1])
        self.activation4 = Sigmoid()
        self.layer4 = [self.affine4, self.activation4]

        self.affine5 = Affine(self.g[4][0], self.g[4][1])
        self.activation5 = Sigmoid()
        self.layer5 = [self.affine5, self.activation5]

        self.layers = [self.layer1, self.layer2, self.layer3, self.layer4, self.layer5]
    def sgd(self, X_train, y_train, lr):
        for i in range(len(X_train)):
            self.dsgd(X_train[i].copy(), y_train[i], lr)
        self.update()
    def train(self, X_train, y_train, epochs, lr):
        for i in range(epochs):
            self.sgd(X_train, y_train, lr)
    def test(self, X_test, y_test):
        cr = 0
        for i in range(len(X_test)):
            cr +=  (self.forward(X_test[i]) >= 0.5) == y_test[i]

        return cr/len(X_test)
    def dsgd(self, x, real, lr):
        xc = x
        ins = []
        outs = [0]
        for layer in self.layers:
            ins.append(x)
            for module in layer:
                x =  module.forward(x)
            outs.append(x)
        diff = x - real

        rolling = diff
        for i in range(4, -1, -1):
            self.g[i][1] -= lr * rolling
            ww = self.g[i][0].copy()
            self.g[i][0] -= lr * np.outer(rolling, ins[i])
            rolling = rolling @ ww
            if i > 0:
                rolling *= (outs[i]) * (1 - outs[i])




        return None
    def __call__(self, x):
        return self.forward(x)

In [14]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(455, 30)
(455, 1)
(114, 30)
(114, 1)


In [15]:
model = MLP_b()

print(model.affine1.weight.shape)

(8, 30)


In [16]:
model = MLP_b()

model.train(X_train, y_train, epochs=100, lr=0.1)

accuracy = model.test(X_test, y_test)
print(accuracy)

[[0.97368421]]


In [17]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the digit data
digits = load_digits()
X = digits.data      # (1797, 64) — flattened 8x8 images
y = digits.target    # (1797,)    — labels 0-9

# Your existing pipeline, unchanged
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y        # optional but recommended
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# One-hot encode labels for your network
Y_train = np.eye(10)[y_train]
Y_test  = np.eye(10)[y_test]

In [18]:
len(X_train[0])

64

In [19]:
class MLP_n:
    def __init__(self):
        self.g = [[np.random.randn(50, 64) * np.sqrt(1/64), np.zeros((1, 50))],
                  [np.random.randn(50, 50) * np.sqrt(1/50), np.zeros((1, 50))],
                  [np.random.randn(50, 50) * np.sqrt(1/50), np.zeros((1, 50))],
                  [np.random.randn(50, 50) * np.sqrt(1/50), np.zeros((1, 50))],
                  [np.random.randn(10, 50) * np.sqrt(1/50), np.zeros((1, 10))]]
        self.update()
    def forward(self, x):
        for layer in self.layers:
            for module in layer:
                x =  module.forward(x)
        return x.astype(float)
    def update(self):
        self.affine1 = Affine(self.g[0][0], self.g[0][1])
        self.activation1 = Sigmoid()
        self.layer1 = [self.affine1, self.activation1]

        self.affine2 = Affine(self.g[1][0], self.g[1][1])
        self.activation2 = Sigmoid()
        self.layer2 = [self.affine2, self.activation2]

        self.affine3 = Affine(self.g[2][0], self.g[2][1])
        self.activation3 = Sigmoid()
        self.layer3 = [self.affine3, self.activation3]

        self.affine4 = Affine(self.g[3][0], self.g[3][1])
        self.activation4 = Sigmoid()
        self.layer4 = [self.affine4, self.activation4]

        self.affine5 = Affine(self.g[4][0], self.g[4][1])
        self.activation5 = Softmax()
        self.layer5 = [self.affine5, self.activation5]

        self.layers = [self.layer1, self.layer2, self.layer3, self.layer4, self.layer5]
    def sgd(self, X_train, y_train, lr):
        for i in range(len(X_train)):
            self.dsgd(X_train[i].copy(), y_train[i], lr)
        self.update()
    def train(self, X_train, y_train, epochs, lr):
        for i in range(epochs):
            self.sgd(X_train, y_train, lr)
    def test(self, X_test, y_test):
        cr = 0
        for i in range(len(X_test)):
            cr +=  (np.argmax(self.forward(X_test[i])) == y_test[i])
        return cr/len(X_test)
    def dsgd(self, x, real, lr):
        xc = x
        ins = []
        outs = [0]
        for layer in self.layers:
            ins.append(x)
            for module in layer:
                x =  module.forward(x)
            outs.append(x)
        diff = x - real

        rolling = diff
        for i in range(4, -1, -1):
            self.g[i][1] -= lr * rolling
            ww = self.g[i][0].copy()
            self.g[i][0] -= lr * np.outer(rolling, ins[i])
            rolling = rolling @ ww
            if i > 0:
                rolling *= (outs[i]) * (1 - outs[i])




        return None
    def __call__(self, x):
        return self.forward(x)

In [20]:
model = MLP_n()

model.train(X_train, Y_train, epochs=100, lr=0.1)

accuracy = model.test(X_test, y_test)
preds = [np.argmax(model.forward(X_test[i])) for i in range(len(X_test))]
print(accuracy)

Streaming output truncated to the last 5000 lines.
5
1
1
8
5
7
5
2
3
7
2
6
5
3
5
0
9
9
8
2
7
6
5
6
1
7
7
5
8
1
1
5
7
0
0
3
3
6
8
7
2
0
0
1
0
4
8
5
8
8
5
9
5
5
1
7
7
6
4
3
3
7
5
9
9
6
7
8
8
2
7
3
3
6
7
6
2
9
5
3
7
1
0
0
2
5
3
7
7
4
5
1
6
5
5
2
1
3
4
4
5
8
8
7
1
3
6
7
3
4
4
8
0
6
9
2
0
9
6
1
5
2
2
7
5
3
4
5
1
4
5
2
0
9
6
7
1
9
2
7
0
0
3
3
0
7
4
5
2
4
8
5
6
7
7
3
8
0
6
1
8
5
4
2
3
7
4
9
5
3
6
0
9
0
7
6
1
4
0
6
8
2
3
0
4
0
6
6
7
6
3
7
1
6
8
0
5
5
0
1
2
8
1
8
3
1
5
5
1
6
6
9
7
7
4
2
8
7
9
6
9
5
8
0
3
7
1
1
5
2
4
4
0
7
1
8
8
1
0
2
8
8
6
7
7
1
7
3
1
4
8
4
1
5
5
3
7
5
8
7
2
9
9
5
8
4
3
5
4
8
5
3
3
9
5
7
5
3
9
2
7
1
2
8
9
4
2
1
9
6
4
8
6
0
4
8
3
4
9
8
7
9
3
9
0
2
4
7
5
2
0
9
9
0
5
1
1
2
8
9
8
3
4
7
2
4
0
9
8
7
6
4
9
3
2
4
6
9
9
1
5
6
3
3
4
9
4
6
2
9
9
4
4
4
3
9
4
9
6
6
1
5
7
3
4
0
5
7
4
9
4
4
2
6
0
3
7
5
6
8
8
3
9
0
6
2
9
9
6
9
9
8
7
2
0
4
8
9
4
0
8
5
8
2
6
6
5
3
0
5
2
0
4
5
2
7
9
2
8
4
2
9
9
8
9
2
3
2
6
0
5
6
2
8
0
4
8
4
3
5
2
0
7
6
6
5
6
6
9
0
2
5
9
4
5
6
1
4
7
7
5
3
5
5
6
2
0
3
7
9
3
8
4
4
5

In [21]:
class MLP_n2:
    def __init__(self):
        self.g = [[np.random.randn(50, 64) * np.sqrt(1/64), np.zeros((1, 50))],
                  [np.random.randn(50, 50) * np.sqrt(1/50), np.zeros((1, 50))],
                  [np.random.randn(50, 50) * np.sqrt(1/50), np.zeros((1, 50))],
                  [np.random.randn(50, 50) * np.sqrt(1/50), np.zeros((1, 50))],
                  [np.random.randn(10, 50) * np.sqrt(1/50), np.zeros((1, 10))]]
        self.update()
    def forward(self, x):
        for layer in self.layers:
            for module in layer:
                x =  module.forward(x)
        return x.astype(float)
    def update(self):
        self.affine = np.empty(5, dtype=object)
        self.activation = np.empty(5, dtype=object)
        self.layer = np.empty(5, dtype=list)
        self.layers = [None] * 5

        for i in range(5):
            self.affine[i] = Affine(self.g[i][0], self.g[i][1])
            if(i != 4):
                self.activation[i] = Sigmoid()
            else:
                self.activation[i] = Softmax()
            self.layer[i] = [self.affine[i], self.activation[i]]
            self.layers[i] = (self.layer[i])



    def sgd(self, X_train, y_train, lr):
        for i in range(len(X_train)):
            self.dsgd(X_train[i].copy(), y_train[i], lr)
        self.update()


    def train(self, X_train, y_train, epochs, lr):
        for i in range(epochs):
            self.sgd(X_train, y_train, lr)


    def test(self, X_test, y_test):
        cr = 0
        for i in range(len(X_test)):
            cr +=  (np.argmax(self.forward(X_test[i])) == y_test[i])

        return cr/len(X_test)



    def dsgd(self, x, real, lr):
        xc = x
        ins = []
        outs = [0]
        for layer in self.layers:
            ins.append(x)
            for module in layer:
                x =  module.forward(x)
            outs.append(x)
        diff = x - real

        rolling = diff
        for i in range(4, -1, -1):
            self.g[i][1] -= lr * rolling
            ww = self.g[i][0].copy()
            self.g[i][0] -= lr * np.outer(rolling, ins[i])
            rolling = rolling @ ww
            if i > 0:
                rolling *= (outs[i]) * (1 - outs[i])
        return None



    def __call__(self, x):
        return self.forward(x)

In [22]:
model = MLP_n2()

model.train(X_train, Y_train, epochs=100, lr=0.1)

accuracy = model.test(X_test, y_test)
preds = [np.argmax(model.forward(X_test[i])) for i in range(len(X_test))]
print(accuracy)

0.9527777777777777


In [29]:
class MLP_n3:
    def __init__(self, arc, out):
        self.g = np.empty(len(arc), dtype=object)
        for i in range(len(arc)):
            if(i != len(arc) - 1):
                self.g[i] = [np.random.randn(arc[i + 1], arc[i]) * np.sqrt(1/arc[i]), np.zeros((1, arc[i + 1]))]
            else:
                self.g[i] = [np.random.randn(out, arc[i]) * np.sqrt(1/arc[i]), np.zeros((1, out))]
        self.update()
    def forward(self, x):
        for layer in self.layers:
            for module in layer:
                x =  module.forward(x)
        return x.astype(float)
    def update(self):
        self.affine = np.empty(5, dtype=object)
        self.activation = np.empty(5, dtype=object)
        self.layer = np.empty(5, dtype=list)
        self.layers = [None] * 5

        for i in range(5):
            self.affine[i] = Affine(self.g[i][0], self.g[i][1])
            if(i != 4):
                self.activation[i] = Sigmoid()
            else:
                self.activation[i] = Softmax()
            self.layer[i] = [self.affine[i], self.activation[i]]
            self.layers[i] = (self.layer[i])



    def sgd(self, X_train, y_train, lr):
        for i in range(len(X_train)):
            self.dsgd(X_train[i].copy(), y_train[i], lr)
        self.update()


    def train(self, X_train, y_train, epochs, lr):
        for i in range(epochs):
            self.sgd(X_train, y_train, lr)


    def test(self, X_test, y_test):
        cr = 0
        for i in range(len(X_test)):
            cr +=  (np.argmax(self.forward(X_test[i])) == y_test[i])

        return cr/len(X_test)



    def dsgd(self, x, real, lr):
        xc = x
        ins = []
        outs = [0]
        for layer in self.layers:
            ins.append(x)
            for module in layer:
                x =  module.forward(x)
            outs.append(x)
        diff = x - real

        rolling = diff
        for i in range(4, -1, -1):
            self.g[i][1] -= lr * rolling
            ww = self.g[i][0].copy()
            self.g[i][0] -= lr * np.outer(rolling, ins[i])
            rolling = rolling @ ww
            if i > 0:
                rolling *= (outs[i]) * (1 - outs[i])
        return None



    def __call__(self, x):
        return self.forward(x)

In [32]:
model = MLP_n3(64, [50, 50, 50], 10)

model.train(X_train, Y_train, epochs=100, lr=0.1)

accuracy = model.test(X_test, y_test)
preds = [np.argmax(model.forward(X_test[i])) for i in range(len(X_test))]
print(accuracy)

0.9527777777777777
